# Information Retrieval. Lab 2: Boolean Retrieval + Vector Space Models

## Introduction

In lab 1 we learnt different methods to calculating an index such as the **binary term-document matrix** and the **term-document count matrix**.

In this lab we'll show how these indexes can be used to retrieve documents using two difference approaches: **Boolean retrieval** and the **Vector Space model**

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

First things first.. we need to create a document index!

In [2]:
# Documents to be indexed, similar to last time:
documents = ['Boats are great',
             'Sailing boats are environmental',
             'canal boats are often called narrow boats, they are quite environmental',
             'wooden boats are more work than metal boats',
             'wooden boats are more work than metal boats, but trawlers are the worst',
             'a wooden house is always the prettiest',
             'narrow cars fit through narrow streets and are environmental',
             'boats boats boats']

In [3]:
vectorizer = CountVectorizer(stop_words='english', binary=True)
term_doc_mat = vectorizer.fit_transform(documents)
vocabulary = vectorizer.get_feature_names_out()
vocabulary

array(['boats', 'called', 'canal', 'cars', 'environmental', 'fit',
       'great', 'house', 'metal', 'narrow', 'prettiest', 'quite',
       'sailing', 'streets', 'trawlers', 'wooden', 'work', 'worst'],
      dtype=object)

In [4]:
# let's just have a look what we have
term_doc_df = pd.DataFrame(term_doc_mat.toarray(), columns=vocabulary)
term_doc_df

,boats,called,canal,cars,environmental,fit,great,house,metal,narrow,prettiest,quite,sailing,streets,trawlers,wooden,work,worst
0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0
2,1,1,1,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0
4,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,1,1
5,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0
6,0,0,0,1,1,1,0,0,0,1,0,0,0,1,0,0,0,0
7,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [5]:
# dictionary that maps each term to the index of the documents its in.
term_occurence = {}
for i, term in enumerate(vocabulary):
  term_occurence[term] = np.where(term_doc_df[term] == 1)[0].tolist()
term_occurence

{'boats': [0, 1, 2, 3, 4, 7],
 'called': [2],
 'canal': [2],
 'cars': [6],
 'environmental': [1, 2, 6],
 'fit': [6],
 'great': [0],
 'house': [5],
 'metal': [3, 4],
 'narrow': [2, 6],
 'prettiest': [5],
 'quite': [2],
 'sailing': [1],
 'streets': [6],
 'trawlers': [4],
 'wooden': [3, 4, 5],
 'work': [3, 4],
 'worst': [4]}

# Boolean Retrieval Model

The **Boolean Retrieval Model** is a classical information retrieval model that operates on the principles of set theory. In this model, documents and queries are represented as sets of terms, and the retrieval process is based on Boolean operations such as AND, OR, and NOT. Each document and query is viewed as a binary vector, where each dimension represents the presence (1) or absence (0) of a specific term. The relevance of documents is determined by the Boolean operators applied to these vectors.



Lets first consider a Boolean retrieval model that handles **only simple 1 term queries**.

In [6]:
def boolean_retrieval_simple(query, term_occurence):

    # Does the query term appear in the vocab
    if query in vocabulary:
      # Which docs does the query appear ?
      results = term_occurence[query]
    else:
      return print('Error: Query term "{}" is not in vocabulary'.format(query))

    return results

# Example simple query with no boolean operators: "environmental"
query = 'environmental'

# Perform boolean retrieval
result = boolean_retrieval_simple(query, term_occurence)

# Print the result
print('Documents matching the query: ')
if result == [] or result==None:
  print('No match')
else:
  for doc_id in result:
    print("Doc ID {}: {}".format(doc_id, documents[doc_id]))

Documents matching the query: 
Doc ID 1: Sailing boats are environmental
Doc ID 2: canal boats are often called narrow boats, they are quite environmental
Doc ID 6: narrow cars fit through narrow streets and are environmental


However Boolean retrieval models use a **set of terms** as the query which are combined using Boolean operators AND, OR, and NOT.

In [7]:
# First lets recap on how we can implement simple boolean operations in python.

x = set(['boats', 'wood', 'river'])
y = set(['canal', 'boats', 'water'])

# X or Y
print('X or Y: ', x.union(y))

# X and Y
print('X and Y: ', x.intersection(y))

# X and not Y.
print('X and not Y: ', x.difference(y))

# Y and not X.
print('Y and not X: ', y.difference(x))

X or Y:  {'water', 'canal', 'wood', 'boats', 'river'}
X and Y:  {'boats'}
X and not Y:  {'river', 'wood'}
Y and not X:  {'water', 'canal'}


Next lets explore a boolean retrieval model, which takes as input a set of queries which are combined using boolean operations, and our term ocurences dictionary. The model retrieves the documents that fit the boolean query.

For simplicity this code can only use the 'AND' boolean operator in its query.

In [8]:
def boolean_retrieval_AND(query, term_occurence):
  # Initialize the result set with all documents
    results_list = []

    # Process the query
    for term in query:
        # Does the query term appear in the vocab
        if term in vocabulary:
          # Which doc does the query appear ?
          doc_idx = term_occurence[term]
          results_list.append(doc_idx) # append it to our results list
        else:
          return print('Error: Query term "{}" is not in vocabulary'.format(term))

    # Find the intersection between all the documents that contain query terms.
    # '*' (astrix) allows us to accept a list of any length (i.e any number of query terms is possible)
    result = set(results_list[0]).intersection(*results_list[1:])

    return list(result)

# Example boolean query: "boats AND environmental"
query = ['boats', 'environmental']

# Perform boolean retrieval
result = boolean_retrieval_AND(query, term_occurence)

# Print the result
print('Documents matching the query: ')
if result == [] or result==None:
  print('No match')
else:
  for doc_id in result:
    print("Doc ID {}: {}".format(doc_id, documents[doc_id]))

Documents matching the query: 
Doc ID 1: Sailing boats are environmental
Doc ID 2: canal boats are often called narrow boats, they are quite environmental


Task: What about a Boolean model that uses the 'OR' operator. Complete the function with your own code

In [9]:
def boolean_retrieval_OR(query, term_occurence):
  # Initialize the result set with all documents
    results_list = []

    # Process the query
    for term in query:
        if term in vocabulary:
          doc_idx = term_occurence[term]
          results_list.append(doc_idx)
        else:
          return print('Error: term "{}" is not in vocabulary'.format(term))

    # Find the union between all the documents that contain query terms.
    # '*' (astrix) allows us to accept a list of any length (i.e any number of query terms is possible)
    result = set(results_list[0]).union(*results_list[1:])

    return list(result)

# Example boolean query: "metal OR environment"
query = ['metal', 'environmental']

# Perform boolean retrieval
result = boolean_retrieval_OR(query, term_occurence)

# Print the result
print('Documents matching the query: ')
if result == []:
  print('No match')
for doc_id in result:
  print("Doc ID {}: {}".format(doc_id, documents[doc_id]))

Documents matching the query: 
Doc ID 1: Sailing boats are environmental
Doc ID 2: canal boats are often called narrow boats, they are quite environmental
Doc ID 3: wooden boats are more work than metal boats
Doc ID 4: wooden boats are more work than metal boats, but trawlers are the worst
Doc ID 6: narrow cars fit through narrow streets and are environmental


What about including the boolean operator 'Not'?

For simplicity, lets consider only AND and NOT boolean queries, for example: **'boats' AND NOT 'environmental'** or **NOT 'streets' AND 'boats**

This one is a little harder.. Here are some hints
* set().difference may be useful here.
* To input the 'NOT' operator, one option would be to include it at the begin of a query term. e.g 'NOTboats'

Make sure you test your function with lots of inputs. Is the output what you expected?

In [10]:
def boolean_retrieval_AND_NOT(query, term_occurence):
  # Initialize the result set with all documents
    results_list = []
    NOT=False

    # Process the query
    for term in query:
        if term.startswith('NOT'):
          term = term[3:]
          NOT=True

        if term in vocabulary:
          doc_idx = term_occurence[term]
          if NOT==True:
            doc_NOT_idx = set(list(range(len(documents)))).difference(set(doc_idx))
            results_list.append(doc_NOT_idx)
          else:
            results_list.append(doc_idx)
        else:
          return print('Error: term "{}" is not in vocabulary'.format(term))

    result = set(results_list[0]).intersection(*results_list[1:])

    return list(result)

# Example boolean query: "boats AND NOT environmental"
query = ['boats', 'NOTenvironmental']

# Perform boolean retrieval
result = boolean_retrieval_AND_NOT(query, term_occurence)

# Print the result
print('Documents matching the query: ')
if result == [] or result==None:
  print('No match')
else:
  for doc_id in result:
    print("Doc ID {}: {}".format(doc_id, documents[doc_id]))

Documents matching the query: 
Doc ID 0: Boats are great
Doc ID 3: wooden boats are more work than metal boats
Doc ID 4: wooden boats are more work than metal boats, but trawlers are the worst
Doc ID 7: boats boats boats


## Vector Space Model

The Vector Space Model (VSM) is a widely used information retrieval model that represents documents and queries as vectors in a multi-dimensional space. In VSM, each term is associated with a dimension, and the weight of a term in a document is proportional to its importance. The relevance of documents to a query is measured based on the similarity of their vectors in this space.

**How do we represent documents and queries as vectors?** Well..we have actually done so already using the binary term-document matrix and the term-document count matrix. The typical approach is using **Tf-IDF** but we'll save that for next week.


In [11]:
# e.g of document vectors

term_doc_mat_np = term_doc_mat.toarray()

d2_vector = term_doc_mat_np[1,:]
print('Doc 2: ', d2_vector)

d4_vector = term_doc_mat_np[3,:]
print('Doc 4: ', d4_vector)



Doc 2:  [1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0]
Doc 4:  [1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 0]


In [12]:
# similar process applies for queries..

def create_query_vector(query, vocabulary):

    # create empty query vector
    query_vector = np.zeros_like(vocabulary)

    # iterate through query terms
    for term in query:
        # if query term is in vocab, then find its position in the vocab array.
        # update the query vector in that position with a 1.
        if term in vocabulary:
            vocab_idx = np.where(vocabulary == term)[0][0]
            query_vector[vocab_idx] = 1
        else:
            print('Error: term "{}" is not in vocabulary'.format(term))

    return query_vector

query = ['boats', 'metal', 'wooden']

query_vector = create_query_vector(query, vocabulary)

print('Query vector: ', query_vector)


Query vector:  [1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0]


**How is similarity measured?** The cosine similarity is used as a measure of how similar two vectors are, and is defnied as follows:


$\text{Cosine Similarity}(\mathbf{d}, \mathbf{q}) = \frac{\mathbf{d} \cdot \mathbf{q}}{\|\mathbf{d}\| \cdot \|\mathbf{q}\|}$

Where d is a document vector and q is a query vector. The output is a score between 0 and 1.

**Task**: Complete the code below for the cosine similarity function

In [13]:
def cosine_similarity(vector1, vector2):

    # numerator = the dot product between d and q
    numerator = np.dot(vector1, vector2)
    # denominator = the magnitude of d multiplied by the magnitude of q
    denominator = np.linalg.norm(vector1) * np.linalg.norm(vector2)

    return numerator / denominator

d2q_similarity = cosine_similarity(d2_vector, query_vector)
print('Cosine similarity between d2 and query: ', d2q_similarity)

d4q_similarity = cosine_similarity(d4_vector, query_vector)
print('Cosine similarity between d4 and query: ', d4q_similarity)



Cosine similarity between d2 and query:  0.33333333333333337
Cosine similarity between d4 and query:  0.8660254037844387


**Question**: Are the outputs what you expected given what you know about docs 2 and 4?

Lets put all the parts together to create the vector space model

In [14]:
def vector_space_model(term_doc_mat_np, query):
  query_vector = create_query_vector(query, vocabulary)

  results = []

  # iterate through all documents
  for doc_idx, doc_vector in enumerate(term_doc_mat_np):
      # calculate the relevance score between doc and query
      relevance_score = cosine_similarity(doc_vector, query_vector)
      results.append([doc_idx, relevance_score, documents[doc_idx]])

  return results

query = ['narrow', 'boats']

results = vector_space_model(term_doc_mat_np, query)

print('Documents sorted by relevance to query:')
results = sorted(results, key=lambda x: x[1], reverse=True)
for res in results:
  print('Doc ID {}, Relevance Score: {:.3f}, Doc: {}'.format(*res))

Documents sorted by relevance to query:
Doc ID 7, Relevance Score: 0.707, Doc: boats boats boats
Doc ID 2, Relevance Score: 0.577, Doc: canal boats are often called narrow boats, they are quite environmental
Doc ID 0, Relevance Score: 0.500, Doc: Boats are great
Doc ID 1, Relevance Score: 0.408, Doc: Sailing boats are environmental
Doc ID 3, Relevance Score: 0.354, Doc: wooden boats are more work than metal boats
Doc ID 6, Relevance Score: 0.316, Doc: narrow cars fit through narrow streets and are environmental
Doc ID 4, Relevance Score: 0.289, Doc: wooden boats are more work than metal boats, but trawlers are the worst
Doc ID 5, Relevance Score: 0.000, Doc: a wooden house is always the prettiest


**Question**: What is different between the output of the boolean retrieval model and the vector space model?

One is ranked with relevance scores, the other is binary (relevance or not relevant)

**Question**: What do you observe in the ranking of the documents? Is there anything that is surprising? Give two observations.

- Longer documents are penalised.
- Documents with repeated works are not given more weight

**Question**: What about if we use the term-document count matrix instead of the binary term-document matrix? Have a think about what changes you expect to see in the ranking of documents. Next, write code to perform vector space retrieval using the count matrix. (You may need to adapt some functions)